In [ ]:
# ============================================================
# INSTALL
# ============================================================

!pip install -q -U \
    transformers \
    accelerate \
    safetensors \
    pandas \
    pillow \
    tqdm \
    matplotlib

In [ ]:
# ============================================================
# DRIVE
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import gc
import json
import math
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm

import matplotlib.pyplot as plt

from transformers import CLIPConfig, CLIPModel, CLIPProcessor
from IPython.display import display

In [ ]:
# ============================================================
# CONFIG
# ============================================================

# Carpeta donde el notebook anterior guardó las imágenes y metadata
GENERATED_ROOT = Path(
    "/content/drive/MyDrive/Proyecto NLP y DL/Imagen_Generacion/generated_images_sd3"
)

# Carpeta nueva para resultados de evaluación
EVAL_ROOT = Path(
    "/content/drive/MyDrive/Proyecto NLP y DL/Imagen_Generacion/evaluation_longclip"
)

EVAL_ROOT.mkdir(parents=True, exist_ok=True)

# Modelo LongCLIP
LONGCLIP_MODEL_ID = "zer0int/LongCLIP-GmP-ViT-L-14"

# Máximo recomendado en la model card de este LongCLIP
LONGCLIP_MAX_LENGTH = 248

# Batch sizes
# Si da OOM, baja IMAGE_BATCH_SIZE a 4 o 2.
IMAGE_BATCH_SIZE = 8
TEXT_BATCH_SIZE = 32

# Runtime
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

# Textos que queremos evaluar contra cada imagen.
# Si una columna no existe para una fila, se ignora.
TEXT_FIELDS_TO_SCORE = [
    "prompt_sd3_large",
    "question",
    "answer_limpio",
    "summary_used",
]

print("DEVICE:", DEVICE)
print("DTYPE:", DTYPE)
print("GENERATED_ROOT:", GENERATED_ROOT)
print("EVAL_ROOT:", EVAL_ROOT)

In [ ]:
# ============================================================
# GENERAL HELPERS
# ============================================================

def clear_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


def load_json(path: Path) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(data: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def save_dataframe(df: pd.DataFrame, name: str) -> None:
    csv_path = EVAL_ROOT / f"{name}.csv"
    json_path = EVAL_ROOT / f"{name}.json"

    df.to_csv(csv_path, index=False)
    df.to_json(json_path, orient="records", force_ascii=False, indent=2)

    print(f"Guardado CSV : {csv_path}")
    print(f"Guardado JSON: {json_path}")


def safe_float(x: Any) -> Optional[float]:
    try:
        if x is None or pd.isna(x):
            return None
        return float(x)
    except Exception:
        return None


def normalize_embeddings(x: torch.Tensor) -> torch.Tensor:
    return F.normalize(x, p=2, dim=-1)

In [ ]:
# ============================================================
# LOAD PREVIOUS GENERATION METADATA
# ============================================================

def find_metadata_files(root: Path) -> List[Path]:
    metadata_files = sorted(root.glob("*/*/metadata.json"))
    return metadata_files


def build_images_dataframe(generated_root: Path) -> pd.DataFrame:
    metadata_files = find_metadata_files(generated_root)

    if not metadata_files:
        raise FileNotFoundError(
            f"No se encontraron metadata.json dentro de: {generated_root}"
        )

    rows = []

    for metadata_path in metadata_files:
        metadata = load_json(metadata_path)

        model_alias = metadata.get("model_alias")
        model_id = metadata.get("model_id")
        source_file = metadata.get("source_file")
        dataset_name = Path(source_file).stem if source_file else metadata_path.parent.name

        records = metadata.get("records", [])

        for rec in records:
            status = rec.get("status")
            image_path = rec.get("image_path")

            if status not in ["ok", "skipped_existing"]:
                continue

            if not image_path:
                continue

            image_path = Path(image_path)

            if not image_path.exists():
                continue

            row_data = rec.get("row_data", {}) or {}

            row = {
                "image_key": f"{model_alias}__{dataset_name}__{rec.get('id')}__{rec.get('row_index')}",
                "model_alias": model_alias,
                "model_id": model_id,
                "source_file": source_file,
                "dataset_name": dataset_name,
                "id": rec.get("id"),
                "row_index": rec.get("row_index"),
                "image_path": str(image_path),
                "generation_status": status,
                "generation_seed": rec.get("seed"),
                "generation_max_sequence_length": rec.get("max_sequence_length"),
                "metadata_path": str(metadata_path),
                "prompt": rec.get("prompt"),
            }

            # Añadimos todo lo que venga de row_data
            for k, v in row_data.items():
                row[k] = v

            # Normalizamos el nombre del prompt principal
            if "prompt_sd3_large" not in row and row.get("prompt") is not None:
                row["prompt_sd3_large"] = row["prompt"]

            rows.append(row)

    df = pd.DataFrame(rows)

    if df.empty:
        raise ValueError("Se han encontrado metadata.json, pero ninguna imagen válida para evaluar.")

    df = df.sort_values(["model_alias", "dataset_name", "row_index"]).reset_index(drop=True)

    return df


df_images = build_images_dataframe(GENERATED_ROOT)

print("Imágenes encontradas:", len(df_images))
display(df_images.head())
print(df_images[["model_alias", "dataset_name"]].value_counts().sort_index())

In [ ]:
# ============================================================
# LOAD LONGCLIP
# ============================================================

def load_longclip(
    model_id: str = LONGCLIP_MODEL_ID,
    max_length: int = LONGCLIP_MAX_LENGTH,
) -> Tuple[CLIPModel, CLIPProcessor]:
    print(f"Cargando LongCLIP: {model_id}")
    print(f"max_position_embeddings: {max_length}")

    start = time.time()

    config = CLIPConfig.from_pretrained(model_id)
    config.text_config.max_position_embeddings = max_length

    model = CLIPModel.from_pretrained(
        model_id,
        config=config,
        torch_dtype=DTYPE,
    )

    processor = CLIPProcessor.from_pretrained(
        model_id,
        padding="max_length",
        max_length=max_length,
    )

    model = model.to(DEVICE)
    model.eval()

    elapsed = time.time() - start
    print(f"LongCLIP cargado en {elapsed:.2f} s")

    return model, processor


longclip_model, longclip_processor = load_longclip()

In [ ]:
# ============================================================
# TOKEN STATS
# ============================================================

def count_longclip_tokens(text: Any) -> Optional[int]:
    if not isinstance(text, str) or not text.strip():
        return None

    tokenized = longclip_processor.tokenizer(
        text,
        add_special_tokens=True,
        truncation=False,
    )

    return len(tokenized["input_ids"])


def token_was_truncated(token_count: Optional[int]) -> Optional[bool]:
    if token_count is None:
        return None
    return token_count > LONGCLIP_MAX_LENGTH


def add_token_stats(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for field in TEXT_FIELDS_TO_SCORE:
        if field not in df.columns:
            continue

        token_col = f"{field}_longclip_token_count"
        trunc_col = f"{field}_longclip_was_truncated"

        tqdm.pandas(desc=f"Tokenizando {field}")

        df[token_col] = df[field].progress_apply(count_longclip_tokens)
        df[trunc_col] = df[token_col].apply(token_was_truncated)

    return df


df_images = add_token_stats(df_images)

display(df_images.head())

In [ ]:
# ============================================================
# EMBEDDING FUNCTIONS
# ============================================================

@torch.inference_mode()
def encode_images(
    image_paths: List[str],
    batch_size: int = IMAGE_BATCH_SIZE,
) -> np.ndarray:
    all_embeddings = []

    for start in tqdm(range(0, len(image_paths), batch_size), desc="Encoding images"):
        batch_paths = image_paths[start:start + batch_size]

        images = []
        for path in batch_paths:
            image = Image.open(path).convert("RGB")
            images.append(image)

        inputs = longclip_processor.image_processor(
            images=images,
            return_tensors="pt",
        )

        pixel_values = inputs["pixel_values"].to(device=DEVICE, dtype=DTYPE)

        image_features = longclip_model.get_image_features(
            pixel_values=pixel_values,
        )

        image_features = normalize_embeddings(image_features)
        all_embeddings.append(image_features.detach().cpu().float().numpy())

        del images, inputs, pixel_values, image_features
        clear_memory()

    return np.vstack(all_embeddings)


@torch.inference_mode()
def encode_texts(
    texts: List[str],
    batch_size: int = TEXT_BATCH_SIZE,
) -> np.ndarray:
    all_embeddings = []

    for start in tqdm(range(0, len(texts), batch_size), desc="Encoding texts"):
        batch_texts = texts[start:start + batch_size]

        inputs = longclip_processor.tokenizer(
            batch_texts,
            padding="max_length",
            max_length=LONGCLIP_MAX_LENGTH,
            truncation=True,
            return_tensors="pt",
        )

        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        text_features = longclip_model.get_text_features(**inputs)
        text_features = normalize_embeddings(text_features)

        all_embeddings.append(text_features.detach().cpu().float().numpy())

        del inputs, text_features
        clear_memory()

    return np.vstack(all_embeddings)


def cosine_from_normalized_embeddings(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    return np.sum(a * b, axis=1)

In [ ]:
# ============================================================
# IMAGE EMBEDDINGS
# ============================================================

image_paths = df_images["image_path"].tolist()

image_embeddings = encode_images(
    image_paths=image_paths,
    batch_size=IMAGE_BATCH_SIZE,
)

print("image_embeddings shape:", image_embeddings.shape)

# Guardamos una copia numpy por si queremos reutilizar
image_embeddings_path = EVAL_ROOT / "longclip_image_embeddings.npy"
np.save(image_embeddings_path, image_embeddings)

print("Embeddings de imagen guardados en:", image_embeddings_path)

In [ ]:
# ============================================================
# LONGCLIP SCORING
# ============================================================

def compute_longclip_scores_for_field(
    df: pd.DataFrame,
    image_embeddings: np.ndarray,
    text_field: str,
) -> pd.DataFrame:
    if text_field not in df.columns:
        print(f"Saltando campo inexistente: {text_field}")
        return pd.DataFrame()

    valid_mask = df[text_field].apply(lambda x: isinstance(x, str) and len(x.strip()) > 0)
    valid_indices = np.where(valid_mask.values)[0]

    if len(valid_indices) == 0:
        print(f"Saltando campo sin textos válidos: {text_field}")
        return pd.DataFrame()

    valid_texts = df.loc[valid_mask, text_field].astype(str).tolist()
    valid_image_embeddings = image_embeddings[valid_indices]

    print("\n" + "=" * 80)
    print(f"Calculando LongCLIP: imagen vs {text_field}")
    print(f"Textos válidos: {len(valid_texts)}")
    print("=" * 80)

    text_embeddings = encode_texts(
        texts=valid_texts,
        batch_size=TEXT_BATCH_SIZE,
    )

    scores = cosine_from_normalized_embeddings(
        valid_image_embeddings,
        text_embeddings,
    )

    rows = []

    token_col = f"{text_field}_longclip_token_count"
    trunc_col = f"{text_field}_longclip_was_truncated"

    for local_i, original_i in enumerate(valid_indices):
        base_row = df.iloc[original_i]

        rows.append({
            "image_key": base_row["image_key"],
            "model_alias": base_row["model_alias"],
            "model_id": base_row["model_id"],
            "source_file": base_row["source_file"],
            "dataset_name": base_row["dataset_name"],
            "id": base_row["id"],
            "row_index": base_row["row_index"],
            "image_path": base_row["image_path"],
            "text_field": text_field,
            "text": base_row[text_field],
            "longclip_score": float(scores[local_i]),
            "longclip_max_length": LONGCLIP_MAX_LENGTH,
            "token_count": base_row.get(token_col, None),
            "was_truncated": base_row.get(trunc_col, None),
        })

    return pd.DataFrame(rows)


score_dfs = []

for text_field in TEXT_FIELDS_TO_SCORE:
    df_field_scores = compute_longclip_scores_for_field(
        df=df_images,
        image_embeddings=image_embeddings,
        text_field=text_field,
    )

    if not df_field_scores.empty:
        score_dfs.append(df_field_scores)

df_scores_long = pd.concat(score_dfs, ignore_index=True)

display(df_scores_long.head())
print("Total scores calculados:", len(df_scores_long))

In [ ]:
# ============================================================
# SAVE LONGCLIP RESULTS
# ============================================================

save_dataframe(df_images, "images_index_with_token_stats")
save_dataframe(df_scores_long, "longclip_scores_long")

# Versión wide: una fila por imagen, una columna por tipo de score
df_scores_wide = df_images.copy()

for text_field in df_scores_long["text_field"].unique():
    tmp = df_scores_long[df_scores_long["text_field"] == text_field][
        ["image_key", "longclip_score", "token_count", "was_truncated"]
    ].copy()

    tmp = tmp.rename(columns={
        "longclip_score": f"longclip_{text_field}_score",
        "token_count": f"longclip_{text_field}_token_count",
        "was_truncated": f"longclip_{text_field}_was_truncated",
    })

    df_scores_wide = df_scores_wide.merge(tmp, on="image_key", how="left")

save_dataframe(df_scores_wide, "longclip_scores_wide")

display(df_scores_wide.head())

In [ ]:
# ============================================================
# SUMMARY BY MODEL AND DATASET
# ============================================================

df_prompt_scores = df_scores_long[
    df_scores_long["text_field"] == "prompt_sd3_large"
].copy()

summary_by_model_dataset = (
    df_prompt_scores
    .groupby(["model_alias", "dataset_name"])
    .agg(
        n=("longclip_score", "count"),
        mean_score=("longclip_score", "mean"),
        std_score=("longclip_score", "std"),
        median_score=("longclip_score", "median"),
        min_score=("longclip_score", "min"),
        max_score=("longclip_score", "max"),
        truncated_pct=("was_truncated", lambda x: float(pd.Series(x).fillna(False).mean())),
        mean_token_count=("token_count", "mean"),
    )
    .reset_index()
    .sort_values(["dataset_name", "model_alias"])
)

save_dataframe(summary_by_model_dataset, "summary_by_model_dataset")

display(summary_by_model_dataset)

In [ ]:
# ============================================================
# PAIRWISE MODEL COMPARISON
# ============================================================

def build_pairwise_comparison(df_prompt_scores: pd.DataFrame) -> pd.DataFrame:
    pivot = (
        df_prompt_scores
        .pivot_table(
            index=["source_file", "dataset_name", "id", "row_index"],
            columns="model_alias",
            values="longclip_score",
            aggfunc="first",
        )
        .reset_index()
    )

    pivot.columns.name = None

    if "sd35_large" in pivot.columns and "sd3_medium" in pivot.columns:
        pivot["delta_sd35_minus_sd3medium"] = pivot["sd35_large"] - pivot["sd3_medium"]

        pivot["winner_longclip"] = np.where(
            pivot["delta_sd35_minus_sd3medium"] > 0,
            "sd35_large",
            np.where(
                pivot["delta_sd35_minus_sd3medium"] < 0,
                "sd3_medium",
                "tie",
            ),
        )
    else:
        print("No se han encontrado ambas columnas: sd35_large y sd3_medium")

    return pivot


df_pairwise = build_pairwise_comparison(df_prompt_scores)

save_dataframe(df_pairwise, "pairwise_sd35_vs_sd3medium_prompt_score")

display(df_pairwise.head())

if "winner_longclip" in df_pairwise.columns:
    display(df_pairwise["winner_longclip"].value_counts(dropna=False))

In [ ]:
# ============================================================
# RANKINGS
# ============================================================

top_n = 20

best_prompt_scores = (
    df_prompt_scores
    .sort_values("longclip_score", ascending=False)
    .head(top_n)
    .reset_index(drop=True)
)

worst_prompt_scores = (
    df_prompt_scores
    .sort_values("longclip_score", ascending=True)
    .head(top_n)
    .reset_index(drop=True)
)

save_dataframe(best_prompt_scores, "top20_best_longclip_prompt_scores")
save_dataframe(worst_prompt_scores, "top20_worst_longclip_prompt_scores")

print("TOP mejores:")
display(best_prompt_scores[[
    "model_alias", "dataset_name", "id", "longclip_score", "token_count", "was_truncated", "image_path"
]])

print("TOP peores:")
display(worst_prompt_scores[[
    "model_alias", "dataset_name", "id", "longclip_score", "token_count", "was_truncated", "image_path"
]])

In [ ]:
# ============================================================
# VISUALIZE RANKED IMAGES
# ============================================================

def make_contact_sheet(
    df: pd.DataFrame,
    title: str,
    score_col: str = "longclip_score",
    max_images: int = 12,
    thumb_size: Tuple[int, int] = (256, 256),
    columns: int = 4,
) -> Image.Image:
    subset = df.head(max_images).copy()

    rows = math.ceil(len(subset) / columns)
    caption_height = 80

    sheet_width = columns * thumb_size[0]
    sheet_height = rows * (thumb_size[1] + caption_height)

    sheet = Image.new("RGB", (sheet_width, sheet_height), "white")
    draw = ImageDraw.Draw(sheet)

    for i, (_, row) in enumerate(subset.iterrows()):
        col = i % columns
        r = i // columns

        x = col * thumb_size[0]
        y = r * (thumb_size[1] + caption_height)

        image = Image.open(row["image_path"]).convert("RGB")
        image.thumbnail(thumb_size)

        paste_x = x + (thumb_size[0] - image.width) // 2
        paste_y = y + (thumb_size[1] - image.height) // 2

        sheet.paste(image, (paste_x, paste_y))

        caption = (
            f"{row['model_alias']} | {row['dataset_name']}\n"
            f"id={row['id']} | score={row[score_col]:.4f}\n"
            f"tokens={row.get('token_count', None)} | trunc={row.get('was_truncated', None)}"
        )

        draw.text(
            (x + 5, y + thumb_size[1] + 5),
            caption,
            fill="black",
        )

    return sheet


best_sheet = make_contact_sheet(
    best_prompt_scores,
    title="Best LongCLIP prompt scores",
    max_images=12,
)

worst_sheet = make_contact_sheet(
    worst_prompt_scores,
    title="Worst LongCLIP prompt scores",
    max_images=12,
)

best_sheet_path = EVAL_ROOT / "contact_sheet_top12_best_longclip.jpg"
worst_sheet_path = EVAL_ROOT / "contact_sheet_top12_worst_longclip.jpg"

best_sheet.save(best_sheet_path, quality=95)
worst_sheet.save(worst_sheet_path, quality=95)

print("Guardado:", best_sheet_path)
display(best_sheet)

print("Guardado:", worst_sheet_path)
display(worst_sheet)

In [ ]:
# ============================================================
# SIDE-BY-SIDE MODEL COMPARISON
# ============================================================

def show_comparison_for_dataset(
    dataset_name: str,
    ids: Optional[List[Any]] = None,
    max_items: int = 8,
    score_field: str = "longclip_prompt_sd3_large_score",
    thumb_size: Tuple[int, int] = (320, 320),
) -> Image.Image:
    df = df_scores_wide[df_scores_wide["dataset_name"] == dataset_name].copy()

    if ids is not None:
        df = df[df["id"].isin(ids)]

    available_ids = (
        df.groupby("id")["model_alias"]
        .nunique()
        .reset_index()
    )

    available_ids = available_ids[available_ids["model_alias"] >= 2]["id"].tolist()
    available_ids = available_ids[:max_items]

    pairs = []

    for item_id in available_ids:
        item = df[df["id"] == item_id].copy()

        row_large = item[item["model_alias"] == "sd35_large"]
        row_medium = item[item["model_alias"] == "sd3_medium"]

        if row_large.empty or row_medium.empty:
            continue

        pairs.append((row_large.iloc[0], row_medium.iloc[0]))

    if not pairs:
        raise ValueError("No se encontraron pares comparables.")

    columns = 2
    rows = len(pairs)
    caption_height = 90

    sheet_width = columns * thumb_size[0]
    sheet_height = rows * (thumb_size[1] + caption_height)

    sheet = Image.new("RGB", (sheet_width, sheet_height), "white")
    draw = ImageDraw.Draw(sheet)

    for r, (large_row, medium_row) in enumerate(pairs):
        for c, row in enumerate([large_row, medium_row]):
            x = c * thumb_size[0]
            y = r * (thumb_size[1] + caption_height)

            image = Image.open(row["image_path"]).convert("RGB")
            image.thumbnail(thumb_size)

            paste_x = x + (thumb_size[0] - image.width) // 2
            paste_y = y + (thumb_size[1] - image.height) // 2

            sheet.paste(image, (paste_x, paste_y))

            score = row.get(score_field, None)
            score_str = f"{score:.4f}" if pd.notna(score) else "NA"

            caption = (
                f"{row['model_alias']}\n"
                f"id={row['id']} | score={score_str}\n"
                f"{row['dataset_name']}"
            )

            draw.text(
                (x + 5, y + thumb_size[1] + 5),
                caption,
                fill="black",
            )

    return sheet


dataset_names = sorted(df_scores_wide["dataset_name"].dropna().unique().tolist())
dataset_names

In [ ]:
# Cambia el dataset_name si quieres ver otro
dataset_name = dataset_names[0]

comparison_sheet = show_comparison_for_dataset(
    dataset_name=dataset_name,
    max_items=8,
)

comparison_path = EVAL_ROOT / f"comparison_sheet_{dataset_name}.jpg"
comparison_sheet.save(comparison_path, quality=95)

print("Guardado:", comparison_path)
display(comparison_sheet)

In [ ]:
# ============================================================
# BASIC PLOTS
# ============================================================

plt.figure(figsize=(10, 5))
df_prompt_scores.boxplot(
    column="longclip_score",
    by=["dataset_name", "model_alias"],
    rot=45,
)
plt.title("LongCLIP prompt-image score by dataset and model")
plt.suptitle("")
plt.ylabel("LongCLIP cosine score")
plt.xlabel("Dataset / model")
plt.tight_layout()

plot_path = EVAL_ROOT / "boxplot_longclip_by_dataset_model.png"
plt.savefig(plot_path, dpi=150)
plt.show()

print("Guardado:", plot_path)

In [ ]:
# ============================================================
# TRUNCATION DIAGNOSTICS
# ============================================================

truncation_summary = (
    df_prompt_scores
    .groupby(["dataset_name", "model_alias"])
    .agg(
        n=("longclip_score", "count"),
        mean_token_count=("token_count", "mean"),
        max_token_count=("token_count", "max"),
        truncated_count=("was_truncated", lambda x: int(pd.Series(x).fillna(False).sum())),
        truncated_pct=("was_truncated", lambda x: float(pd.Series(x).fillna(False).mean())),
        mean_score=("longclip_score", "mean"),
    )
    .reset_index()
    .sort_values(["dataset_name", "model_alias"])
)

save_dataframe(truncation_summary, "truncation_summary_prompt")

display(truncation_summary)

In [ ]:
# ============================================================
# TOKEN COUNT VS LONGCLIP SCORE
# ============================================================

tmp = df_prompt_scores.dropna(subset=["token_count", "longclip_score"]).copy()

correlations = (
    tmp
    .groupby(["dataset_name", "model_alias"])
    .apply(
        lambda g: pd.Series({
            "n": len(g),
            "pearson_token_score": g["token_count"].corr(g["longclip_score"], method="pearson"),
            "spearman_token_score": g["token_count"].corr(g["longclip_score"], method="spearman"),
        })
    )
    .reset_index()
)

save_dataframe(correlations, "correlation_token_count_vs_longclip_score")

display(correlations)

plt.figure(figsize=(8, 5))
plt.scatter(tmp["token_count"], tmp["longclip_score"])
plt.xlabel("LongCLIP token count")
plt.ylabel("LongCLIP prompt-image score")
plt.title("Token count vs LongCLIP score")
plt.tight_layout()

scatter_path = EVAL_ROOT / "scatter_token_count_vs_longclip_score.png"
plt.savefig(scatter_path, dpi=150)
plt.show()

print("Guardado:", scatter_path)

In [ ]:
# ============================================================
# FREE VRAM
# ============================================================

try:
    del longclip_model
    del longclip_processor
except Exception:
    pass

try:
    del image_embeddings
except Exception:
    pass

clear_memory()

print("LongCLIP liberado de memoria.")